In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import roc_auc_score, precision_recall_curve
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, precision_recall_curve
)

In [2]:
train_df = pd.read_csv("../data/train_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")

In [3]:
features = [
    'T2M', 'RH2M', 'PS',
    'hour', 'month',
    'T2M_diff', 'RH2M_diff', 'PS_diff',
    'T2M_roll_mean', 'RH2M_roll_mean', 'PS_roll_mean',
    'T2M_roll_std', 'RH2M_roll_std', 'PS_roll_std',
    'T2M_roll_dev', 'RH2M_roll_dev', 'PS_roll_dev'
]

X_train = train_df[features]

X_test = test_df[features]
y_test = test_df['is_anomaly']

In [4]:
y_true = test_df['is_anomaly']

In [5]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.03,
    random_state=42
)

model.fit(X_train)

,n_estimators,100
,max_samples,'auto'
,contamination,0.03
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [6]:
y_pred = (model.predict(X_test) == -1).astype(int)

In [7]:
test_df['spike_candidate'] = (
    abs(test_df['T2M_diff']) > 8
)

In [8]:
spikes = test_df[
    test_df['anomaly_type'] == 'temperature_spike'
]

normal = test_df[
    test_df['anomaly_type'] == 'normal'
]

print(
    "Spike detected:",
    spikes['spike_candidate'].mean() * 100,
    "%"
)

print(
    "Normal falsely flagged:",
    normal['spike_candidate'].mean() * 100,
    "%"
)

Spike detected: 100.0 %
Normal falsely flagged: 1.7080745341614907 %


In [9]:
y_true_spike = (test_df['anomaly_type'] == 'temperature_spike').astype(int)
y_pred_spike = test_df['spike_candidate'].astype(int)

In [10]:
test_df = test_df.sort_values('datetime').reset_index(drop=True)

In [11]:
def get_recent_temps(df, index, window=6):
    start = max(0, index - window)
    return df['T2M'].iloc[start:index].values
    

In [12]:
test_df['recent_std_6'] = (
    test_df['T2M']
    .rolling(6)
    .std()
    .shift(1)
)

In [13]:
test_df['frozen_candidate'] = (
    test_df['recent_std_6'] < 0.05
)

In [14]:
y_true_frozen = (test_df['anomaly_type'] == 'temperature_frozen').astype(int)
y_pred_frozen = test_df['frozen_candidate'].astype(int)

In [15]:
from sklearn.linear_model import LinearRegression

def fit_residual_model(df, target, predictors):
    model = LinearRegression()
    model.fit(df[predictors], df[target])
    return model

resid_models = {
    'T2M': fit_residual_model(train_df, 'T2M', ['RH2M', 'PS']),
    'RH2M': fit_residual_model(train_df, 'RH2M', ['T2M', 'PS']),
    'PS': fit_residual_model(train_df, 'PS', ['T2M', 'RH2M']),
}

resid_stds = {}
for target, predictors in [('T2M', ['RH2M','PS']), ('RH2M', ['T2M','PS']), ('PS', ['T2M','RH2M'])]:
    pred = resid_models[target].predict(train_df[predictors])
    resid_stds[target] = (train_df[target] - pred).std()

print(resid_stds)

{'T2M': 4.5796470168636105, 'RH2M': 19.312599515054053, 'PS': 3.6929143684632195}


In [16]:
y_true_multi = (
    test_df['anomaly_type'] == 'multivariate_inconsistency'
).astype(int)

In [17]:
for target, predictors in [('T2M', ['RH2M','PS']), ('RH2M', ['T2M','PS']), ('PS', ['T2M','RH2M'])]:
    pred = resid_models[target].predict(test_df[predictors])
    test_df[f'{target}_resid_z'] = (test_df[target] - pred) / resid_stds[target]

test_df['multivariate_score_resid'] = (
    test_df['T2M_resid_z']**2 +
    test_df['RH2M_resid_z']**2 +
    test_df['PS_resid_z']**2
)

auc_resid = roc_auc_score(y_true_multi, test_df['multivariate_score_resid'])
print("ROC-AUC (residual-based):", auc_resid)

ROC-AUC (residual-based): 0.7182568734648601


In [18]:
# quick check that all key variables exist after a full run
for name in ['train_df', 'test_df', 'features', 'model', 'y_true', 'y_pred', 'y_true_spike', 'y_pred_spike',
             'y_true_frozen', 'y_pred_frozen', 'y_true_multi', 'resid_models', 'resid_stds', 'multi_predictors']:
    print(name, '->', 'OK' if name in dir() else 'MISSING')

train_df -> OK
test_df -> OK
features -> OK
model -> OK
y_true -> OK
y_pred -> OK
y_true_spike -> OK
y_pred_spike -> OK
y_true_frozen -> OK
y_pred_frozen -> OK
y_true_multi -> OK
resid_models -> OK
resid_stds -> OK
multi_predictors -> MISSING


In [19]:
precision_r, recall_r, thresholds_r = precision_recall_curve(
    y_true_multi, test_df['multivariate_score_resid']
)

f1_r = 2 * precision_r * recall_r / (precision_r + recall_r + 1e-9)
best_idx_r = np.argmax(f1_r)

print("Best threshold:", thresholds_r[best_idx_r])
print("At that threshold -> precision:", precision_r[best_idx_r], "recall:", recall_r[best_idx_r], "f1:", f1_r[best_idx_r])


Best threshold: 17.07574477523787
At that threshold -> precision: 0.24 recall: 0.32 f1: 0.2742857137959183


In [20]:
test_df['multivariate_candidate'] = (
    test_df['multivariate_score_resid'] > 17.0757
)

print("MULTIVARIATE INCONSISTENCY (residual-based)")
print("Accuracy:", accuracy_score(y_true_multi, test_df['multivariate_candidate'].astype(int)))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_multi, test_df['multivariate_candidate'].astype(int)))
print("\nClassification Report:")
print(classification_report(y_true_multi, test_df['multivariate_candidate'].astype(int)))

MULTIVARIATE INCONSISTENCY (residual-based)
Accuracy: 0.9703409621672116

Confusion Matrix:
[[4131   76]
 [  51   24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4207
           1       0.24      0.32      0.27        75

    accuracy                           0.97      4282
   macro avg       0.61      0.65      0.63      4282
weighted avg       0.97      0.97      0.97      4282



In [21]:
normal_rows = test_df[test_df['anomaly_type'] == 'normal']

fp_rate = normal_rows['multivariate_candidate'].mean()
print("False positive rate on normal rows:", fp_rate * 100, "%")
print("Normal rows flagged:", normal_rows['multivariate_candidate'].sum(), "out of", len(normal_rows))


False positive rate on normal rows: 0.07763975155279502 %
Normal rows flagged: 3 out of 3864


In [22]:
multi_predictors = {
    'T2M': ['RH2M', 'PS'],
    'RH2M': ['T2M', 'PS'],
    'PS': ['T2M', 'RH2M'],
}

In [23]:
# Compute residual z-scores for train_df too (currently only test_df has them)
for target, predictors in [('T2M', ['RH2M','PS']), ('RH2M', ['T2M','PS']), ('PS', ['T2M','RH2M'])]:
    pred = resid_models[target].predict(train_df[predictors])
    train_df[f'{target}_resid_z'] = (train_df[target] - pred) / resid_stds[target]

features_v2 = features + ['T2M_resid_z', 'RH2M_resid_z', 'PS_resid_z']

X_train_v2 = train_df[features_v2]
X_test_v2 = test_df[features_v2]

model_v2 = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)
model_v2.fit(X_train_v2)

y_pred_v2 = (model_v2.predict(X_test_v2) == -1).astype(int)

report_v2 = classification_report(y_true, y_pred_v2, output_dict=True)
print(f"With residual features -> precision={report_v2['1']['precision']:.3f}, recall={report_v2['1']['recall']:.3f}, f1={report_v2['1']['f1-score']:.3f}")
print("\nOriginal (17 features) for comparison -> f1=0.371")

With residual features -> precision=0.465, recall=0.435, f1=0.450

Original (17 features) for comparison -> f1=0.371


In [50]:
%%writefile detector.py
import numpy as np


def pd_isna(val):
    import pandas as pd
    return pd.isna(val)


def _scale_confidence(value, threshold, higher_is_worse=True):
    if higher_is_worse:
        excess = (value - threshold) / threshold
    else:
        excess = (threshold - value) / threshold
    excess = max(0.0, excess)
    confidence = 50 + 50 * min(1.0, excess)
    return round(confidence, 1)


# Physically plausible ranges - adjust if your actual data range differs
VALID_RANGES = {
    'T2M': (-60, 60),      # Celsius
    'RH2M': (0, 100),      # percent
    'PS': (850, 1100),     # hPa, sea-level-ish range
}


def _clamp(target, value):
    lo, hi = VALID_RANGES[target]
    return max(lo, min(hi, value))


class SkyGuardDetector:
    def __init__(self, model, features, resid_models, resid_stds, multi_predictors,
                 multi_threshold=17.0757, if_score_min=-0.09150155162021423, if_score_max=0.1269836965997279):
        self.model = model
        self.features = features
        self.resid_models = resid_models
        self.resid_stds = resid_stds
        self.multi_predictors = multi_predictors
        self.multi_threshold = multi_threshold
        self.if_score_min = if_score_min
        self.if_score_max = if_score_max

    def _predict_healed_reading(self, row):
        healed = {}
        for target, predictors in self.multi_predictors.items():
            pred_val = self.resid_models[target].predict(
                row[predictors].values.reshape(1, -1)
            )[0]
            healed[target] = _clamp(target, float(pred_val))
        return healed

    def _safe_heal_with_missing(self, row):
        healed = {}
        for target in ['T2M', 'RH2M', 'PS']:
            if not pd_isna(row[target]):
                healed[target] = float(row[target])
                continue
            predictors = self.multi_predictors[target]
            if row[predictors].isna().any():
                healed[target] = None
            else:
                pred_val = self.resid_models[target].predict(
                    row[predictors].values.reshape(1, -1)
                )[0]
                healed[target] = _clamp(target, float(pred_val))
        return healed

    def detect(self, row, recent_temps):
        raw_reading = {
            'T2M': None if pd_isna(row['T2M']) else float(row['T2M']),
            'RH2M': None if pd_isna(row['RH2M']) else float(row['RH2M']),
            'PS': None if pd_isna(row['PS']) else float(row['PS']),
        }

        if row[['T2M', 'RH2M', 'PS']].isna().any():
            return {
                'status': 'anomaly',
                'type': 'communication_error',
                'severity': 'high',
                'confidence': 100.0,
                'reason': 'Missing sensor reading',
                'raw_reading': raw_reading,
                'healed_reading': self._safe_heal_with_missing(row)
            }

        if abs(row['T2M_diff']) > 8:
            confidence = _scale_confidence(abs(row['T2M_diff']), 8, higher_is_worse=True)
            return {
                'status': 'anomaly',
                'type': 'temperature_spike',
                'severity': 'high',
                'confidence': confidence,
                'reason': 'Temperature changed suddenly',
                'raw_reading': raw_reading,
                'healed_reading': self._predict_healed_reading(row)
            }

        if len(recent_temps) >= 6:
            recent_std = recent_temps[-6:].std()
            if recent_std < 0.05:
                confidence = _scale_confidence(recent_std, 0.05, higher_is_worse=False)
                return {
                    'status': 'anomaly',
                    'type': 'temperature_frozen',
                    'severity': 'medium',
                    'confidence': confidence,
                    'reason': 'Temperature barely changed for 6 readings',
                    'raw_reading': raw_reading,
                    'healed_reading': self._predict_healed_reading(row)
                }

        z_scores = {}
        multi_score = 0
        for target, predictors in self.multi_predictors.items():
            pred_val = self.resid_models[target].predict(
                row[predictors].values.reshape(1, -1)
            )[0]
            z = (row[target] - pred_val) / self.resid_stds[target]
            z_scores[target] = z
            multi_score += z ** 2

        if multi_score > self.multi_threshold:
            confidence = _scale_confidence(multi_score, self.multi_threshold, higher_is_worse=True)
            return {
                'status': 'anomaly',
                'type': 'multivariate_inconsistency',
                'severity': 'medium',
                'confidence': confidence,
                'reason': 'Sensor readings individually normal but jointly inconsistent',
                'raw_reading': raw_reading,
                'healed_reading': self._predict_healed_reading(row)
            }

        base_features = row[self.features].values
        resid_features = np.array([z_scores['T2M'], z_scores['RH2M'], z_scores['PS']])
        x = np.concatenate([base_features, resid_features]).reshape(1, -1)

        pred = self.model.predict(x)[0]
        score = self.model.decision_function(x)[0]

        if pred == -1:
            if score < 0:
                confidence = 100 * (0 - score) / (0 - self.if_score_min)
                confidence = round(min(100.0, max(0.0, confidence)), 1)
            else:
                confidence = 50.0
            return {
                'status': 'anomaly',
                'type': 'ml_anomaly',
                'severity': 'medium',
                'confidence': confidence,
                'reason': 'Unusual weather-sensor pattern detected',
                'raw_reading': raw_reading,
                'healed_reading': self._predict_healed_reading(row)
            }

        return {
            'status': 'normal',
            'type': 'normal',
            'severity': 'none',
            'confidence': None,
            'reason': 'No abnormal behaviour detected',
            'raw_reading': raw_reading,
            'healed_reading': None
        }

#renew

Overwriting detector.py


In [51]:
import importlib
import detector
importlib.reload(detector)
from detector import SkyGuardDetector

detector_v2 = SkyGuardDetector(
    model=model_v2,
    features=features,
    resid_models=resid_models,
    resid_stds=resid_stds,
    multi_predictors=multi_predictors,
    if_score_min=if_score_min,
    if_score_max=if_score_max
)

joblib.dump(detector_v2, '../models/skyguard_detector_v2.joblib')
loaded_detector = joblib.load('../models/skyguard_detector_v2.joblib')

test_row = test_df[test_df['anomaly_type'] == 'temperature_spike'].iloc[0]
idx = test_df[test_df['anomaly_type'] == 'temperature_spike'].index[0]
recent = get_recent_temps(test_df, idx)
print(loaded_detector.detect(test_row, recent))

{'status': 'anomaly', 'type': 'temperature_spike', 'severity': 'high', 'confidence': 100.0, 'reason': 'Temperature changed suddenly', 'raw_reading': {'T2M': 46.98042714991526, 'RH2M': 29.57, 'PS': 992.0}, 'healed_reading': {'T2M': 20.839074437362797, 'RH2M': 0, 'PS': 972.4370175269138}}


In [27]:
from sklearn.ensemble import IsolationForest

for c in [0.02, 0.03, 0.05, 0.08, 0.10, 0.15]:
    temp_model = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    temp_model.fit(X_train)
    temp_pred = temp_model.predict(X_test)
    temp_pred_binary = (temp_pred == -1).astype(int)

    report = classification_report(y_true, temp_pred_binary, output_dict=True)
    print(f"contamination={c} -> precision={report['1']['precision']:.3f}, recall={report['1']['recall']:.3f}, f1={report['1']['f1-score']:.3f}")

contamination=0.02 -> precision=0.384, recall=0.356, f1=0.370
contamination=0.03 -> precision=0.348, recall=0.397, f1=0.371
contamination=0.05 -> precision=0.280, recall=0.447, f1=0.344
contamination=0.08 -> precision=0.232, recall=0.507, f1=0.319
contamination=0.1 -> precision=0.208, recall=0.538, f1=0.300
contamination=0.15 -> precision=0.175, recall=0.610, f1=0.272


In [28]:
normal_row = test_df[test_df['anomaly_type'] == 'normal'].iloc[0]
idx = test_df[test_df['anomaly_type'] == 'normal'].index[0]
recent_normal = get_recent_temps(test_df, idx)

result_normal = loaded_detector.detect(normal_row, recent_normal)
print(result_normal)

{'status': 'anomaly', 'type': 'temperature_spike', 'severity': 'high', 'reason': 'Temperature changed suddenly', 'raw_reading': {'T2M': 25.4, 'RH2M': 30.23, 'PS': 991.3}, 'healed_reading': {'T2M': 21.463529871741684, 'RH2M': 30.942562119787453, 'PS': 986.8332688766136}}


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [29]:
for idx in test_df[test_df['anomaly_type'] == 'normal'].index:
    row = test_df.loc[idx]
    recent = get_recent_temps(test_df, idx)
    result = loaded_detector.detect(row, recent)
    if result['type'] in ('ml_anomaly', 'normal'):
        print("Found row at index:", idx)
        print(result)
        break

Found row at index: 2
{'status': 'normal', 'type': 'normal', 'severity': 'none', 'reason': 'No abnormal behaviour detected', 'raw_reading': {'T2M': 24.33, 'RH2M': 34.76, 'PS': 991.1}, 'healed_reading': None}


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/p

In [52]:
import warnings
warnings.filterwarnings('ignore')

predicted_types = []
confidences = []

for i in range(len(test_df)):
    row = test_df.iloc[i]
    recent = get_recent_temps(test_df, i)
    result = loaded_detector.detect(row, recent)
    predicted_types.append(result['type'])
    confidences.append(result['confidence'])

test_df['predicted_type'] = predicted_types
test_df['confidence'] = confidences

print("Any nulls in predicted_type?", test_df['predicted_type'].isna().any())
print("\npredicted_type value counts:")
print(test_df['predicted_type'].value_counts())

print("\nConfidence stats by predicted type:")
print(test_df.groupby('predicted_type')['confidence'].describe())

Any nulls in predicted_type? False

predicted_type value counts:
predicted_type
normal                        3801
ml_anomaly                     247
temperature_spike              143
temperature_frozen              53
multivariate_inconsistency      38
Name: count, dtype: int64

Confidence stats by predicted type:
                            count        mean        std    min     25%  \
predicted_type                                                            
ml_anomaly                  247.0   20.547368  18.918765    0.1    6.40   
multivariate_inconsistency   38.0   75.823684  21.014582   50.0   54.95   
normal                        0.0         NaN        NaN    NaN     NaN   
temperature_frozen           53.0  100.000000   0.000000  100.0  100.00   
temperature_spike           143.0   97.231469   9.324717   53.7  100.00   

                              50%    75%    max  
predicted_type                                   
ml_anomaly                   15.7   29.1  100.0  
multiv

In [32]:
import subprocess
result = subprocess.run(
    ['grep', '-n', 'detect_status', '07_model_training_ipynb.ipynb'],
    capture_output=True, text=True
)
print(result.stdout)

In [33]:
for name in ['train_df', 'test_df', 'features', 'features_v2', 'model', 'model_v2',
             'y_true', 'y_test', 'y_pred', 'y_pred_v2',
             'y_true_spike', 'y_pred_spike', 'y_true_frozen', 'y_pred_frozen', 'y_true_multi',
             'resid_models', 'resid_stds', 'multi_predictors']:
    print(name, '->', 'OK' if name in dir() else 'MISSING')

print("\npredicted_type in test_df columns?", 'predicted_type' in test_df.columns)

train_df -> OK
test_df -> OK
features -> OK
features_v2 -> OK
model -> OK
model_v2 -> OK
y_true -> OK
y_test -> OK
y_pred -> OK
y_pred_v2 -> OK
y_true_spike -> OK
y_pred_spike -> OK
y_true_frozen -> OK
y_pred_frozen -> OK
y_true_multi -> OK
resid_models -> OK
resid_stds -> OK
multi_predictors -> OK

predicted_type in test_df columns? True


In [41]:
with open('detector.py') as f:
    content = f.read()
print('_safe_heal_with_missing' in content)
print("for target in ['T2M', 'RH2M', 'PS']" in content)

True
True


In [42]:
import detector
print(detector.__file__)

/Users/himanshu/Documents/my documents/hackathon/aws-anomally-detection/ml/detector.py


In [44]:
# Test missing RH2M
test_row_rh2m = test_df.iloc[10].copy()
test_row_rh2m['RH2M'] = np.nan
recent = get_recent_temps(test_df, 10)
result_rh2m = loaded_detector.detect(test_row_rh2m, recent)
print("Missing RH2M:", result_rh2m)

# Test missing PS
test_row_ps = test_df.iloc[10].copy()
test_row_ps['PS'] = np.nan
result_ps = loaded_detector.detect(test_row_ps, recent)
print("\nMissing PS:", result_ps)

# Test two missing at once (T2M and RH2M both gone)
test_row_two = test_df.iloc[10].copy()
test_row_two['T2M'] = np.nan
test_row_two['RH2M'] = np.nan
result_two = loaded_detector.detect(test_row_two, recent)
print("\nMissing T2M and RH2M:", result_two)

Missing RH2M: {'status': 'anomaly', 'type': 'communication_error', 'severity': 'high', 'reason': 'Missing sensor reading', 'raw_reading': {'T2M': 13.5, 'RH2M': None, 'PS': 991.8}, 'healed_reading': {'T2M': 13.5, 'RH2M': 60.6279444511365, 'PS': 991.8}}

Missing PS: {'status': 'anomaly', 'type': 'communication_error', 'severity': 'high', 'reason': 'Missing sensor reading', 'raw_reading': {'T2M': 13.5, 'RH2M': 63.93, 'PS': None}, 'healed_reading': {'T2M': 13.5, 'RH2M': 63.93, 'PS': 990.9668060010813}}

Missing T2M and RH2M: {'status': 'anomaly', 'type': 'communication_error', 'severity': 'high', 'reason': 'Missing sensor reading', 'raw_reading': {'T2M': None, 'RH2M': None, 'PS': 991.8}, 'healed_reading': {'T2M': None, 'RH2M': None, 'PS': 991.8}}


In [46]:
if_scores_train = model_v2.decision_function(X_train_v2)
if_score_min = float(if_scores_train.min())
if_score_max = float(if_scores_train.max())
print("IF score range:", if_score_min, "to", if_score_max)

IF score range: -0.09150155162021423 to 0.1269836965997279
